# TP final · Opción B · Efecto de la música

## Análisis exploratorio

Problema: predecir `Music effects` (`Improve`, `No effect`, `Worsen`) a partir de los hábitos de escucha reportados en la encuesta.

Este notebook es el análisis exploratorio. Carga la encuesta con Spark y la describe, sin imputar, sin descartar filas y sin entrenar. El objetivo es decidir después qué arreglo merece la tabla. Para eso mira:

- Cuántas filas y columnas hay, y qué significa cada columna
- Qué valores toma cada columna, cuántos hay de cada uno y cuántos faltan
- En `Age`, `Hours per day` y `BPM`, el mínimo, el máximo y si hay vacíos o números que no se pueden leer
- Qué filas de `BPM` quedan fuera de `40–250`. Ese rango no es una regla de la encuesta: sirve para ver tempos imposibles, como `0` o `999999999`.

Con esa salida se elige el arreglo. Los modelos y qué puntajes de salud entran como variables quedan para el notebook de entrenamiento.

**Fuente**: [Music & Mental Health Survey Results](https://www.kaggle.com/datasets/catherinerasgaitis/mxmh-survey-results), Catherine Rasgaitis, licencia CC0. El CSV está en `data/mxmh_survey_results.csv`.


## Widgets

- **Archivo CSV.** Ruta a los datos de la encuesta. Ya viene con la ruta del CSV de este repo.

In [ ]:
def ruta_csv_default():
    notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    carpeta = "/Workspace" + notebook.rsplit("/", 1)[0]
    return "file:" + carpeta + "/data/mxmh_survey_results.csv"

for nombre in ("csv_path", "puntajes_salud", "puntajes_algunas"):
    try:
        dbutils.widgets.remove(nombre)
    except Exception:
        pass

dbutils.widgets.text("csv_path", ruta_csv_default(), "Archivo CSV")


In [ ]:
from pyspark.sql import functions as F

def resolver_csv():
    pedido = dbutils.widgets.get("csv_path").strip()
    return pedido or ruta_csv_default()

csv_path = resolver_csv()
print("csv:", csv_path)

crudo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)
print(f"filas: {crudo.count():,}  columnas: {len(crudo.columns)}")


## 1. Qué hay en la tabla

Conteos, nulos y las tres clases. Una celda vacía cuenta como faltante, igual que un nulo. En cada columna se listan hasta 8 valores, ordenados por frecuencia. No se rellena nada.


In [ ]:
display(crudo.limit(5))

TOPE = 8
filas = []
for nombre in crudo.columns:
    texto = F.trim(F.col(nombre).cast("string"))
    vacio = F.col(nombre).isNull() | (texto == "")
    presentes = crudo.where(~vacio)
    conteos = (
        presentes.groupBy(texto.alias("valor"))
        .count()
        .orderBy(F.desc("count"), "valor")
        .collect()
    )
    mostrados = conteos[:TOPE]
    partes = [f"{fila.valor} ({fila['count']})" for fila in mostrados]
    resto = len(conteos) - len(mostrados)
    if resto > 0:
        partes.append(f"y {resto} más")
    filas.append((nombre, int(crudo.where(vacio).count()), len(conteos), ", ".join(partes)))

display(spark.createDataFrame(filas, ["columna", "nulos", "distintos", "valores"]))


In [ ]:
etiqueta = F.when(
    F.col("Music effects").isNull() | (F.trim(F.col("Music effects")) == ""),
    "(vacio)",
).otherwise(F.col("Music effects"))
display(
    crudo.groupBy(etiqueta.alias("Music effects"))
    .count()
    .orderBy(F.desc("count"))
)


## 2. Age, horas y BPM

Resumen numérico y las filas de BPM vacías, no numéricas o fuera de 40–250. Ese rango no viene de la encuesta ni de la consigna: un tema suele ir de 60 a 180, y 40–250 es un margen ancho para marcar valores que no son un tempo, como 0 o 999999999. No decide el arreglo. Esta celda no modifica la tabla.


In [ ]:
def resumen_numerico(nombre):
    texto = F.trim(F.col(nombre).cast("string"))
    numero = F.col(nombre).cast("double")
    vacio = F.col(nombre).isNull() | (texto == "")
    no_numerico = ~vacio & numero.isNull()
    print(nombre)
    display(crudo.select(
        F.count("*").alias("filas"),
        F.count(F.when(vacio, 1)).alias("vacios"),
        F.count(F.when(no_numerico, 1)).alias("no_numerico"),
        F.min(numero).alias("minimo"),
        F.max(numero).alias("maximo"),
    ))

for nombre in ("Age", "Hours per day", "BPM"):
    resumen_numerico(nombre)

bpm = F.col("BPM").cast("double")


In [ ]:
raros = crudo.where(
    F.col("BPM").isNull()
    | (F.trim(F.col("BPM").cast("string")) == "")
    | bpm.isNull()
    | (bpm < 40)
    | (bpm > 250)
)
print(f"filas con BPM vacio, no numerico o fuera de 40-250: {raros.count()}")
display(raros.select("Age", "Fav genre", "BPM", "Music effects").orderBy(bpm.asc_nulls_first()))


## Modelos previstos

Cuando el arreglo esté decidido, los cuatro ven la misma matriz y el mismo corte 80/20 estratificado (`random_state=7`). Los tres de árboles usan el balanceo de clases. Gana el mayor F1 macro de test.

| Run | Modelo |
|---|---|
| `logistica` | Regresión logística, `C=1` |
| `random_forest` | Random forest |
| `lightgbm` | LightGBM |
| `catboost` | CatBoost |

El entrenamiento queda para después de mirar esta exploración. No se imputa en este notebook todavía.

Qué puntajes de salud entran como variables se elige en ese notebook, no en este.


## Qué significa cada columna

Los textos de frecuencia son `Never`, `Rarely`, `Sometimes` y `Very frequently`. Los puntajes de salud van de 0 a 10: 0 es no experimentarlo y 10 es experimentarlo de forma constante o extrema. Son autodeclarados, no un diagnóstico.

| Columna | Qué es |
|---|---|
| `Timestamp` | Fecha y hora en que envió el formulario |
| `Age` | Edad |
| `Primary streaming service` | Servicio de música que más usa |
| `Hours per day` | Horas de música por día |
| `While working` | Si escucha música mientras trabaja |
| `Instrumentalist` | Si toca un instrumento |
| `Composer` | Si compone música |
| `Fav genre` | Género favorito |
| `Exploratory` | Si busca artistas o géneros nuevos |
| `Foreign languages` | Si escucha música en otros idiomas |
| `BPM` | Pulsos por minuto que declaró |
| `Frequency [Classical]` | Con qué frecuencia escucha classical |
| `Frequency [Country]` | Con qué frecuencia escucha country |
| `Frequency [EDM]` | Con qué frecuencia escucha EDM |
| `Frequency [Folk]` | Con qué frecuencia escucha folk |
| `Frequency [Gospel]` | Con qué frecuencia escucha gospel |
| `Frequency [Hip hop]` | Con qué frecuencia escucha hip hop |
| `Frequency [Jazz]` | Con qué frecuencia escucha jazz |
| `Frequency [K pop]` | Con qué frecuencia escucha K-pop |
| `Frequency [Latin]` | Con qué frecuencia escucha latin |
| `Frequency [Lofi]` | Con qué frecuencia escucha lofi |
| `Frequency [Metal]` | Con qué frecuencia escucha metal |
| `Frequency [Pop]` | Con qué frecuencia escucha pop |
| `Frequency [R&B]` | Con qué frecuencia escucha R&B |
| `Frequency [Rap]` | Con qué frecuencia escucha rap |
| `Frequency [Rock]` | Con qué frecuencia escucha rock |
| `Frequency [Video game music]` | Con qué frecuencia escucha música de videojuegos |
| `Anxiety` | Puntaje de ansiedad |
| `Depression` | Puntaje de depresión |
| `Insomnia` | Puntaje de insomnio |
| `OCD` | Puntaje de trastorno obsesivo-compulsivo |
| `Music effects` | Si la música le mejora (`Improve`), no le hace nada (`No effect`) o le empeora (`Worsen`) |
| `Permissions` | Si aceptó que se publique su respuesta |
